# UK Retail Sales Analysis

**Portfolio project — Pavan Shetty**

This notebook analyses a synthetic UK retail transaction dataset using Python and pandas. The dataset is deliberately synthetic and contains no real customer information.

## Business questions
1. What are the headline sales and profitability KPIs?
2. How does revenue change through the year?
3. Which product categories and products contribute the most revenue?
4. Which UK regions perform strongest?
5. How do customer segments differ?
6. What effect does discounting have on profitability?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw" / "retail_sales.csv"
PROCESSED = ROOT / "data" / "processed" / "retail_sales_clean.csv"

df_raw = pd.read_csv(RAW)
df_raw.head()


## 1. Data quality assessment


In [ ]:
print("Shape:", df_raw.shape)
print("\nDuplicates by Order_ID:", df_raw["Order_ID"].duplicated().sum())
print("\nMissing values:")
display(df_raw.isna().sum()[df_raw.isna().sum() > 0])


The raw dataset intentionally includes a small number of duplicated transactions, missing city/payment values and whitespace issues. This allows the project to demonstrate a realistic cleaning workflow.


## 2. Cleaning and feature engineering


In [ ]:
df = df_raw.copy()

text_cols = [
    "Customer_Name", "Product_Name", "Category", "Region",
    "City", "Payment_Method", "Customer_Segment"
]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

df["City"] = df["City"].fillna("Unknown")
df["Payment_Method"] = df["Payment_Method"].fillna("Unknown")

df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df = df.dropna(subset=["Order_Date"])
df = df.drop_duplicates(subset=["Order_ID"], keep="first")

numeric_cols = ["Quantity","Unit_Price","Discount","Revenue","Cost","Profit"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=numeric_cols)
df = df[(df["Quantity"] > 0) & (df["Unit_Price"] > 0)]

df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.strftime("%b")
df["Quarter"] = "Q" + df["Order_Date"].dt.quarter.astype(str)
df["Profit_Margin_Pct"] = np.where(
    df["Revenue"] != 0,
    df["Profit"] / df["Revenue"] * 100,
    0
)

df = df.sort_values(["Order_Date","Order_ID"]).reset_index(drop=True)
df.to_csv(PROCESSED, index=False)

print("Clean shape:", df.shape)
print("Remaining duplicate Order_IDs:", df["Order_ID"].duplicated().sum())
print("Remaining missing values:", int(df.isna().sum().sum()))


## 3. Executive KPIs


In [ ]:
total_revenue = df["Revenue"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order_ID"].nunique()
total_customers = df["Customer_ID"].nunique()
aov = total_revenue / total_orders
profit_margin = total_profit / total_revenue * 100

kpis = pd.DataFrame({
    "Metric": ["Revenue","Profit","Orders","Customers","Average Order Value","Profit Margin %"],
    "Value": [
        f"£{total_revenue:,.2f}",
        f"£{total_profit:,.2f}",
        f"{total_orders:,}",
        f"{total_customers:,}",
        f"£{aov:,.2f}",
        f"{profit_margin:.2f}%"
    ]
})
kpis


## 4. Monthly sales trend


In [ ]:
monthly = (
    df.groupby(df["Order_Date"].dt.to_period("M"))
      .agg(Revenue=("Revenue","sum"), Profit=("Profit","sum"), Orders=("Order_ID","nunique"))
      .reset_index()
)
monthly["Month"] = monthly["Order_Date"].astype(str)

plt.figure(figsize=(10,5))
plt.plot(monthly["Month"], monthly["Revenue"], marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

monthly.sort_values("Revenue", ascending=False).head()


## 5. Product category performance


In [ ]:
category_perf = (
    df.groupby("Category")
      .agg(
          Revenue=("Revenue","sum"),
          Profit=("Profit","sum"),
          Orders=("Order_ID","nunique"),
          Units=("Quantity","sum")
      )
      .assign(Profit_Margin_Pct=lambda x: x["Profit"] / x["Revenue"] * 100)
      .sort_values("Revenue", ascending=False)
)

category_perf


In [ ]:
plot_df = category_perf.reset_index()

plt.figure(figsize=(9,5))
plt.bar(plot_df["Category"], plot_df["Revenue"])
plt.title("Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 6. Regional performance


In [ ]:
regional = (
    df.groupby("Region")
      .agg(
          Revenue=("Revenue","sum"),
          Profit=("Profit","sum"),
          Orders=("Order_ID","nunique")
      )
      .assign(AOV=lambda x: x["Revenue"] / x["Orders"])
      .sort_values("Revenue", ascending=False)
)

regional


In [ ]:
plot_df = regional.head(8).sort_values("Revenue")

plt.figure(figsize=(9,5))
plt.barh(plot_df.index, plot_df["Revenue"])
plt.title("Top Regions by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Region")
plt.tight_layout()
plt.show()


## 7. Top products


In [ ]:
top_products = (
    df.groupby(["Product_Name","Category"])
      .agg(Revenue=("Revenue","sum"), Profit=("Profit","sum"), Units=("Quantity","sum"))
      .sort_values("Revenue", ascending=False)
      .head(10)
)

top_products


## 8. Customer segment analysis


In [ ]:
segment = (
    df.groupby("Customer_Segment")
      .agg(
          Revenue=("Revenue","sum"),
          Profit=("Profit","sum"),
          Orders=("Order_ID","nunique"),
          Customers=("Customer_ID","nunique")
      )
      .assign(
          Revenue_Per_Customer=lambda x: x["Revenue"] / x["Customers"],
          AOV=lambda x: x["Revenue"] / x["Orders"]
      )
      .sort_values("Revenue", ascending=False)
)

segment


## 9. Discount impact


In [ ]:
discount = (
    df.groupby("Discount")
      .agg(Revenue=("Revenue","sum"), Profit=("Profit","sum"), Orders=("Order_ID","nunique"))
      .assign(Profit_Margin_Pct=lambda x: x["Profit"] / x["Revenue"] * 100)
)

discount


## 10. Business findings

Using the generated dataset (seed 42):

- Total revenue is approximately **£7,679,530** from **20,000** unique orders.
- Total profit is approximately **£2,625,729**, giving a profit margin of about **34.2%**.
- **2025-12** is the highest-revenue month, reflecting the intentional year-end seasonal uplift in the synthetic data.
- **Electronics** is the highest-revenue category.
- **London** is the highest-revenue region.
- **Laptop Pro 14** is the highest-revenue individual product.
- **100.0%** of customers placed more than one order.

Because the dataset is synthetic, these findings demonstrate analytical technique rather than claims about the real UK retail market.


## 11. Recommended business actions

1. Use the strongest seasonal months for inventory and campaign planning.
2. Protect margin when applying larger discounts; compare revenue lift with profit-margin erosion.
3. Prioritise high-value products and regions while investigating low-performing areas for assortment, pricing or acquisition issues.
4. Track customer segment value using revenue per customer and average order value, not revenue alone.
5. Extend the project with a star-schema Power BI model, cohort retention analysis or forecasting.
